# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [5]:
ПРІЗВИЩЕ_ІМЯ = 'Голейчук Іван'
ГРУПА        = 'Ші-33'
НОМЕР        = 2

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [6]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 2   ·   Голейчук Іван   ·   Ші-33

Ваші вісім стовпців:
   1. Телефон
   2. Супутній
   3. ЛН
   4. Дата розтину
   5. КТ
   6. Фібрин1
   7. Сечов2
   8. ФЕРИТ1

Ваша пара для аналізу зв’язку: ЛН та помер/жив

Ваше додаткове питання:
   Стовпець «АТ пост» містить значення на кшталт «140/100». Запропонуйте три способи перетворити його на числові ознаки та обґрунтуйте, який із них візьмете в модель.


In [7]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Супутній,ЛН,Дата розтину,КТ,Фібрин1,Сечов2,ФЕРИТ1
0,0958469788 дружина Ганна,"ІХС, КС, СН1, НВ ФК2",ІІІ-IV,2022-03-11,20-30%,4,6.5,NaN
1,"0671778486, 0994887663 дружина Оксана","ЦД, ІІ тип",ІІ-ІІІ,NaT,30-35%,5,4,839.0
2,"0958768045, чоловік Степан 0951784726","ІХС. КС, ГХІІ, ГЛШ СНІІа, порушення толерантності до глю...",ІІ,NaT,70-80%,5.29,10.1,NaN
3,0959819331. дочка Валентина 0960845205,"ЦД, ІІ тип, ІХС, Диф кардіоск, стенокар напруги, синусов...",ІІІ-IV,NaT,0.8,2.5,12.5,NaN
4,"0952294866, дочка Дарина 0678360161",ІХС ДК ГХІІ-ІІІ СНІІа ГЛШст3,ІІ-ІІІ,NaT,NaN,4.29,7.4,NaN
5,0959922983 сестра Софія,"ІХС ДК ГХІІ ГЛШ СНІІ ФК2, ЦД ІІ",ІІ-ІІІ,2021-12-26,30-40%,5,21.4,NaN
6,"0998073702, дружина Наталія 0998382226","ІХС ДК ГХ ІІа ФК3, ЦД вперше виявлений",ІІ-ІІІ,NaT,60-65%,3.24,6.1,NaN
7,"0637258771, син Сергій 0968413301","ІХС ДК ГХІІ ГЛШ СНІІа ФК3, полімедик алергія,ерозивий г/...",І,NaT,до 17%,5.04,7.6,NaN


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [8]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Телефон       120 non-null    object        
 1   Супутній      115 non-null    str           
 2   ЛН            121 non-null    str           
 3   Дата розтину  61 non-null     datetime64[us]
 4   КТ            44 non-null     object        
 5   Фібрин1       115 non-null    object        
 6   Сечов2        102 non-null    object        
 7   ФЕРИТ1        2 non-null      float64       
dtypes: datetime64[us](1), float64(1), object(4), str(2)
memory usage: 7.7+ KB


In [9]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Супутній
    непорожніх 115 з 121 | унікальних 91 | типи: str×115
    приклади: ["'ІХС, КС, СН1, НВ ФК2'", "'ЦД, ІІ тип'", "'ІХС. КС, ГХІІ, ГЛШ СНІІа, порушення толерантності до глюкози'", "'ЦД, ІІ тип, ІХС, Диф кардіоск, стенокар напруги, синусова тахік, ГХІІ, СНІІв '"]

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20-30%'", "'30-35%'", "'7

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [10]:
ПАСПОРТ = {
    'Телефон': ('ідентифікатор', 'це контактні номери родичів із підписами, а не величина: порівнювати, додавати чи усереднювати їх не має сенсу'),
    'Супутній': ('номінальна', 'вільний текст зі списком супутніх діагнозів: значення можна лише порівняти на рівність, порядку й відстаней між ними немає'),
    'ЛН': ('порядкова', 'ступінь легеневої недостатності І < ІІ < ІІІ < IV має природний порядок, але різниця між сусідніми ступенями невідома й нерівна'),
    'Дата розтину': ('інтервальна', 'різниці між датами мають сенс (дні між подіями), а нуль календаря умовний: початок відліку обрано за домовленістю'),
    'КТ': ('відношень', 'частка ураженої легеневої тканини: є природний нуль (0% - уражень немає), тому 40% справді удвічі більше за 20%'),
    'Фібрин1': ('відношень', 'лабораторна концентрація при першому аналізі: нуль означає відсутність речовини, тому відношення значень мають сенс'),
    'Сечов2': ('відношень', 'концентрація сечовини в ммоль/л при другому аналізі: природний нуль, тому можна казати, що значення удвічі більше'),
    'ФЕРИТ1': ('відношень', 'концентрація феритину з природним нулем, але заповнена лише у двох пацієнтів зі 121, тож статистики за нею недостовірні'),
    'помер/жив': ('номінальна', 'коди 1 і 2 - лише мітки двох результатів лікування, а не кількості: порядку й відстані між ними немає'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,"це контактні номери родичів із підписами, а не величина:..."
1,Супутній,номінальна,вільний текст зі списком супутніх діагнозів: значення мо...
2,ЛН,порядкова,ступінь легеневої недостатності І < ІІ < ІІІ < IV має пр...
3,Дата розтину,інтервальна,"різниці між датами мають сенс (дні між подіями), а нуль ..."
4,КТ,відношень,частка ураженої легеневої тканини: є природний нуль (0% ...
5,Фібрин1,відношень,лабораторна концентрація при першому аналізі: нуль означ...
6,Сечов2,відношень,концентрація сечовини в ммоль/л при другому аналізі: при...
7,ФЕРИТ1,відношень,"концентрація феритину з природним нулем, але заповнена л..."


In [11]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [12]:
import re

ПОРЯДОК = {'ЛН': ['І', 'І-ІІ', 'ІІ', 'ІІ-ІІІ', 'ІІІ', 'ІІІ-IV']}


def числові_значення(стовпець):
    def розбір(v):
        if isinstance(v, (int, float, np.integer)):
            return float(v)
        знайдено = [float(x) for x in re.findall(r'\d+(?:\.\d+)?', str(v).replace(',', '.'))]
        if not знайдено:
            return np.nan
        x = sum(знайдено) / len(знайдено)
        return x / 100 if стовпець == 'КТ' else x
    return df[стовпець].dropna().map(розбір).dropna()


def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        р['усі значення унікальні'] = з.nunique() == len(з)
        return р

    if ш in ('номінальна', 'порядкова'):
        частоти = з.astype(str).str.strip().value_counts()
        р['мода'] = частоти.index[0]
        р['найчастіші'] = частоти.head(3).to_dict()
        if ш == 'порядкова':
            порядок = ПОРЯДОК[стовпець]
            коди = pd.Categorical(з.astype(str).str.strip(), categories=порядок, ordered=True).codes
            р['медіана'] = порядок[int(np.quantile(коди, 0.5, method='lower'))]
            р['квантилі 25% і 75%'] = (порядок[int(np.quantile(коди, 0.25, method='lower'))],
                                       порядок[int(np.quantile(коди, 0.75, method='lower'))])
        return р

    if pd.api.types.is_datetime64_any_dtype(df[стовпець]):
        д = pd.to_datetime(з)
        р['мінімум'] = д.min().date()
        р['максимум'] = д.max().date()
        р['медіана'] = д.median().date()
        р['середня дата'] = д.mean().date()
        р['стандартне відхилення, днів'] = round((д - д.min()).dt.days.std(), 1)
        return р

    ч = числові_значення(стовпець)
    р['числових значень'] = len(ч)
    р['мода'] = ч.mode().iloc[0] if ч.value_counts().iloc[0] > 1 else 'немає'
    р['медіана'] = round(ч.median(), 3)
    р['квантилі 25% і 75%'] = (round(ч.quantile(0.25), 3), round(ч.quantile(0.75), 3))
    р['середнє'] = round(ч.mean(), 3)
    р['стандартне відхилення'] = round(ч.std(), 3)
    if ш in ('відношень', 'абсолютна'):
        р['коефіцієнт варіації, %'] = round(100 * ч.std() / ч.mean(), 1)
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- Телефон  (ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120, 'усі значення унікальні': True}

--- Супутній  (номінальна)
    {'непорожніх': 115, 'унікальних': 91, 'мода': 'ІХС ДК ГХст2', 'найчастіші': {'ІХС ДК ГХст2': 7, 'ІХС.ДК. ГХ ІІ': 5, 'ІХС ДК ГХст2.': 4}}

--- ЛН  (порядкова)
    {'непорожніх': 121, 'унікальних': 6, 'мода': 'ІІ-ІІІ', 'найчастіші': {'ІІ-ІІІ': 59, 'ІІ': 21, 'ІІІ': 17}, 'медіана': 'ІІ-ІІІ', 'квантилі 25% і 75%': ('ІІ', 'ІІ-ІІІ')}

--- Дата розтину  (інтервальна)
    {'непорожніх': 61, 'унікальних': 50, 'мінімум': datetime.date(2021, 1, 6), 'максимум': datetime.date(2022, 6, 22), 'медіана': datetime.date(2022, 4, 23), 'середня дата': datetime.date(2022, 4, 6), 'стандартне відхилення, днів': np.float64(72.6)}

--- КТ  (відношень)
    {'непорожніх': 44, 'унікальних': 26, 'числових значень': 43, 'мода': np.float64(0.3), 'медіана': np.float64(0.35), 'квантилі 25% і 75%': (np.float64(0.262), np.float64(0.638)), 'середнє': np.float64(0.425), 'стандартне відхилен

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [13]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Супутній,115,91,ІХС ДК ГХст2,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Фібрин1,115.0,87.0,6.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Сечов2,102.0,66.0,5.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ФЕРИТ1,2.0,NaN,NaN,NaN,555.05,271.1,413.075,555.05,697.025,839.0,401.565941


In [14]:
тел_числа = pd.to_numeric(df['Телефон'], errors='coerce')
код_лн = pd.Categorical(df['ЛН'], categories=ПОРЯДОК['ЛН'], ordered=True).codes + 1
розтини = df['Дата розтину'].dropna()
від_епохи = (розтини - pd.Timestamp('1970-01-01')).dt.days
від_2021 = (розтини - pd.Timestamp('2021-01-01')).dt.days
cv_епоха = 100 * від_епохи.std() / від_епохи.mean()
cv_2021 = 100 * від_2021.std() / від_2021.mean()
кт_числа = pd.to_numeric(df['КТ'], errors='coerce')
ферит = pd.to_numeric(df['ФЕРИТ1'], errors='coerce')

БЕЗГЛУЗДІ = [
    ('Телефон', 'середнє', round(тел_числа.mean()),
     f'середнє береться лише за {тел_числа.notna().sum()} номерами, збереженими як числа (без початкового нуля), а решта {df["Телефон"].notna().sum() - тел_числа.notna().sum()} записів є текстом із іменами родичів; номер телефону - ідентифікатор, а не величина'),
    ('ЛН', 'середнє кодів 1-6', round(код_лн.mean(), 2),
     'порядкова шкала: коди 1-6 лише нумерують ступені, відстань між І та ІІ не дорівнює відстані між ІІІ та IV, тому «середній ступінь» залежить від довільної нумерації'),
    ('Дата розтину', 'коефіцієнт варіації, %', round(cv_епоха, 3),
     f'інтервальна шкала з умовним нулем: дні лічать від 1970-01-01, а при відліку від 2021-01-01 те саме CV дорівнює {cv_2021:.1f}%, тобто число визначається вибором нуля, а не даними'),
    ('ФЕРИТ1', 'середнє', round(ферит.mean(), 2),
     f'змінна кількісна, але заповнена лише у {ферит.notna().sum()} пацієнтів зі {len(df)}, тому середнє двох значень нічого не каже про вибірку'),
    ('КТ', 'середнє', round(кт_числа.mean(), 3),
     f'стовпець змішаного формату: to_numeric залишив лише {кт_числа.notna().sum()} часток із {df["КТ"].notna().sum()} заповнених, а діапазони на кшталт «70-80%» мовчки втрачено, тож середнє описує лише частину записів'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,середнє,7.587990e+08,"середнє береться лише за 23 номерами, збереженими як чис..."
1,ЛН,середнє кодів 1-6,3.570000e+00,"порядкова шкала: коди 1-6 лише нумерують ступені, відста..."
2,Дата розтину,"коефіцієнт варіації, %",3.800000e-01,інтервальна шкала з умовним нулем: дні лічать від 1970-0...
3,ФЕРИТ1,середнє,5.550500e+02,"змінна кількісна, але заповнена лише у 2 пацієнтів зі 12..."
4,КТ,середнє,3.960000e-01,стовпець змішаного формату: to_numeric залишив лише 27 ч...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [15]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: ЛН × помер/жив

ЛН: шкала — порядкова
помер/жив: шкала — номінальна

       ЛН  помер/жив
0  ІІІ-IV          1
1  ІІ-ІІІ          2
2      ІІ          2
3  ІІІ-IV          2
4  ІІ-ІІІ          2
5  ІІ-ІІІ          1


In [16]:
from scipy.stats import chi2_contingency

ОБРАНА_МІРА = 'коефіцієнт Крамера V (на основі χ²)'
ЧОМУ_САМЕ_ВОНА = ('ЛН - порядкова шкала, а помер/жив - номінальна (бінарна), тому за таблицею підходить '
                  'коефіцієнт Крамера V: він не вимагає числових відстаней між ступенями й не залежить від кодів.')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

таблиця = pd.crosstab(df[A], df[B]).reindex(ПОРЯДОК['ЛН'])
таблиця.columns = ['помер', 'вижив']
хі2, p, ступені_вільності, очікувані = chi2_contingency(таблиця)
n = таблиця.values.sum()
V = np.sqrt(хі2 / (n * (min(таблиця.shape) - 1)))

print(таблиця.to_string())
print()
print(f'χ² = {хі2:.2f}, ступенів вільності = {ступені_вільності}, p = {p:.2g}, n = {n}')
print(f'Коефіцієнт Крамера V = {V:.3f}')
print(f'Клітинок з очікуваною частотою менше 5: {(очікувані < 5).sum()} з {очікувані.size}')
print()
print(f'Зв’язок сильний: V = {V:.2f}. Усі померлі мали ЛН не нижче ІІ-ІІІ, а при ЛН до ІІ включно не помер ніхто.')
print('Через малі очікувані частоти p-значення наближене, але величина V від цього не залежить.')

        помер  вижив
ЛН                  
І           0     12
І-ІІ        0      9
ІІ          0     21
ІІ-ІІІ     49     10
ІІІ        11      6
ІІІ-IV      1      2

χ² = 69.58, ступенів вільності = 5, p = 1.3e-13, n = 121
Коефіцієнт Крамера V = 0.758
Клітинок з очікуваною частотою менше 5: 4 з 12

Зв’язок сильний: V = 0.76. Усі померлі мали ЛН не нижче ІІ-ІІІ, а при ЛН до ІІ включно не помер ніхто.
Через малі очікувані частоти p-значення наближене, але величина V від цього не залежить.


In [17]:
помер = (df[B] == 1).astype(int)


def кореляція(коди):
    return np.corrcoef(коди, помер)[0, 1]


правильні_коди = pd.Categorical(df[A], categories=ПОРЯДОК['ЛН'], ordered=True).codes
коди_за_появою = pd.factorize(df[A])[0]
ген = np.random.default_rng(0)
випадкові = [кореляція(ген.permutation(6)[правильні_коди]) for _ in range(1000)]

print(f'Пірсон, коди за природним порядком 0-5    : r = {кореляція(правильні_коди):.3f}')
print(f'Пірсон, коди за порядком появи в таблиці  : r = {кореляція(коди_за_появою):.3f}')
print(f'Пірсон, 1000 випадкових нумерацій ступенів: r від {min(випадкові):.3f} до {max(випадкові):.3f}')
print()
print(f'Ті самі люди й ті самі ступені, а r змінюється від {min(випадкові):.2f} до {max(випадкові):.2f}, залежно від того, яке число')
print('присвоєно якому ступеню. Число визначається нумерацією, а не даними, тож інтерпретувати його не можна.')
print('Навіть за правильного порядку Пірсон припускає однакові відстані між ступенями, яких порядкова шкала не гарантує.')

Пірсон, коди за природним порядком 0-5    : r = 0.544
Пірсон, коди за порядком появи в таблиці  : r = -0.395
Пірсон, 1000 випадкових нумерацій ступенів: r від -0.737 до 0.739

Ті самі люди й ті самі ступені, а r змінюється від -0.74 до 0.74, залежно від того, яке число
присвоєно якому ступеню. Число визначається нумерацією, а не даними, тож інтерпретувати його не можна.
Навіть за правильного порядку Пірсон припускає однакові відстані між ступенями, яких порядкова шкала не гарантує.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [18]:
ПІДГОТОВКА = {
    'Телефон': 'у модель не брати узагалі: ідентифікатор і персональні дані родичів, прогностичної цінності не має',
    'Супутній': 'вільний текст розбити на окремі діагнози (ІХС, гіпертонія, діабет, СН, ХОЗЛ) і закодувати бінарними прапорцями (one-hot), порядок не вводити',
    'ЛН': 'впорядкована категорія із заданим порядком І < І-ІІ < ІІ < ІІ-ІІІ < ІІІ < ІІІ-IV, цілі коди 0-5 лише за цим порядком',
    'Дата розтину': 'перейти від дати до тривалості в днях від поступлення; для прогнозу летальності не брати, бо заповнена лише у померлих (витік цілі)',
    'КТ': 'привести до єдиної частки 0-1 (діапазон замінити серединою), далі стандартизація; велику частку пропусків вирішувати окремо',
    'Фібрин1': 'логарифмування через скошений розподіл із викидами й стандартизація (z-оцінка), пропуски позначити окремо',
    'Сечов2': 'логарифмування через сильну правосторонню скошеність (максимум 116) і стандартизація',
    'ФЕРИТ1': 'у модель не брати: заповнено лише у двох пацієнтів зі 121, нормалізація такої ознаки не має сенсу',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,у модель не брати узагалі: ідентифікатор і персональні д...
1,Супутній,номінальна,"вільний текст розбити на окремі діагнози (ІХС, гіпертоні..."
2,ЛН,порядкова,впорядкована категорія із заданим порядком І < І-ІІ < ІІ...
3,Дата розтину,інтервальна,перейти від дати до тривалості в днях від поступлення; д...
4,КТ,відношень,привести до єдиної частки 0-1 (діапазон замінити середин...
5,Фібрин1,відношень,логарифмування через скошений розподіл із викидами й ста...
6,Сечов2,відношень,логарифмування через сильну правосторонню скошеність (ма...
7,ФЕРИТ1,відношень,у модель не брати: заповнено лише у двох пацієнтів зі 12...


In [19]:
готові = pd.DataFrame(index=df.index)

діагнози = {'ІХС': 'ІХС', 'ГХ': 'ГХ|гіпертонічна', 'ЦД': 'ЦД|діабет', 'СН': 'СН', 'ХОЗЛ': 'ХОЗЛ'}
текст = df['Супутній'].astype('string')
for назва, шаблон in діагнози.items():
    готові['супутній_' + назва] = текст.str.contains(шаблон, regex=True).astype('Float64')

готові['ЛН_код'] = pd.Categorical(df['ЛН'], categories=ПОРЯДОК['ЛН'], ordered=True).codes

тривалість = (df['Дата розтину'] - pd.to_datetime(df['Поступлення'], errors='coerce')).dt.days
готові['днів_до_розтину'] = тривалість.where(тривалість >= 0)

фібрин = pd.to_numeric(df['Фібрин1'], errors='coerce')
лог_фібрин = np.log(фібрин)
готові['Фібрин1_лог_z'] = (лог_фібрин - лог_фібрин.mean()) / лог_фібрин.std()

print('Від’ємних тривалостей до розтину (помилки в датах), замінено на пропуск:', int((тривалість < 0).sum()))
print()
print(готові.astype(float).describe().T.round(3).to_string())

Від’ємних тривалостей до розтину (помилки в датах), замінено на пропуск: 2

                 count    mean     std    min    25%     50%     75%     max
супутній_ІХС     115.0   0.852   0.356  0.000  1.000   1.000   1.000   1.000
супутній_ГХ      115.0   0.609   0.490  0.000  0.000   1.000   1.000   1.000
супутній_ЦД      115.0   0.252   0.436  0.000  0.000   0.000   0.500   1.000
супутній_СН      115.0   0.287   0.454  0.000  0.000   0.000   1.000   1.000
супутній_ХОЗЛ    115.0   0.043   0.205  0.000  0.000   0.000   0.000   1.000
ЛН_код           121.0   2.570   1.196  0.000  2.000   3.000   3.000   5.000
днів_до_розтину   59.0  17.220  14.637  3.000  8.000  15.000  21.500  80.000
Фібрин1_лог_z    114.0  -0.000   1.000 -6.266 -0.408   0.119   0.581   3.057


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [20]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [21]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [22]:
ВИПРАВЛЕННЯ = [
    (1,
     'Середнє арифметичне для номінальної (бінарної) змінної: коди 1 і 2 - лише мітки результату лікування',
     'Значення 1.50 каже тільки те, що померлих і вижилих майже порівну (61 і 60); це залежить від довільно обраних кодів і не означає «середній стан між летальним і сприятливим результатом».',
     'Подавати частоти й частки: 61 померлий (50.4%) і 60 вижилих, мода - померлі; середнє для цієї змінної не рахувати.'),
    (2,
     'Дані непридатні: одну категорію записано по-різному («ж» і «жін», «ч» і «чол»), а nunique() рахує написання, а не статі',
     'Чотири «статі» - це дві статі в різних записах, тому висновок про чотири статі хибний, а частка 46% для «ж» занижена: насправді жінок 75 із 121.',
     'Спершу звести написання до двох значень (ч - 46, ж - 75), потім рахувати частоти: жінки 62%, чоловіки 38%.'),
    (3,
     'Середнє для порядкової шкали, до того ж порядок задано алфавітом (sort_index), а написання не очищено',
     'Алфавіт ставить «важкий» перед «легкий», а «важкий » із пробілом окремою категорією, тому «середній ступінь» - це позиція в алфавітному списку з дублями, а не тяжкість; «легкий» у висновку безглуздий.',
     'Очистити написання, задати порядок легкий < середній < важкий < вкрай важкий < критичний і рахувати частоти та медіану: медіана - «середній».'),
    (4,
     'Дані непридатні: КТ має змішаний формат (частки 0.3, рядки «70-80%», «до 17%», порожні рядки), а to_numeric(errors=coerce) мовчки відкинув текстові значення',
     'Середнє 39.6% пораховано лише за 27 числовими записами з 44 заповнених (та ще 77 зі 121 значень відсутні), тож воно залежить від формату, у якому внесли дані, і не описує всіх пацієнтів.',
     'Привести всі записи до єдиної частки (діапазон - середина, «до 17%» - окреме рішення) і подавати медіану разом із кількістю використаних значень та часткою пропусків.'),
    (5,
     'Дані непридатні: у spO2 без дотації є значення 0.9 - ймовірно, помилка введення (замість 90)',
     'Живий пацієнт не може мати сатурацію 0.9%; цей викид тягне середнє вниз (82.05 проти 82.73 без нього) і роздуває стандартне відхилення, а мінімум 0.9 у звіті прийнято за реальний.',
     'Перевірити первинний запис, виправити або виключити викид і подавати медіану (84) разом із міжквартильним розмахом, а мінімум - лише після очищення.'),
    (6,
     'Коефіцієнт варіації для інтервальної шкали (дати): нуль відліку умовний',
     'Дати переведено в дні від 1970-01-01, тому середнє близько 19000 днів і CV мізерне (0.59%) через штучно великий базис; при відліку від іншої дати CV інше, тож «менше 1%» нічого не каже про однорідність потоку.',
     'Для дат подавати діапазон, медіану та стандартне відхилення в днях; CV не рахувати, а рівномірність потоку оцінювати за кількістю госпіталізацій на тиждень.'),
    (7,
     'Кореляція Пірсона між двома номінальними змінними (стать і результат) із довільними кодами 0 і 1',
     'r = 0.177 залежить від того, які числа дали категоріям, а «близьке до нуля» неправда: χ² = 3.78, p = 0.052, зв’язок на межі значущості, тож відсутність зв’язку не доведена.',
     'Таблиця спряженості, χ² та коефіцієнт Крамера V (для 2×2 дорівнює 0.18), з висновком, що результат неоднозначний і на цих даних потребує більшої вибірки.'),
    (8,
     'Мін-макс нормалізація застосована до всіх числових стовпців підряд, включно з міткою помер/жив і майже порожніми та сталими ознаками',
     'ФЕРИТ2 має єдине значення, max - min = 0, тож ділення на нуль дало NaN у всьому стовпці; ФЕРИТ1 нормалізовано за двома значеннями, а помер/жив - номінальна мітка; твердження «усі значення в [0, 1]» хибне.',
     'Спершу відібрати кількісні ознаки за шкалою й варіативністю (без сталих, майже порожніх, ідентифікаторів і цільової змінної), нормалізувати лише їх і перевіряти max > min.'),
    (9,
     'Дані непридатні: dropna() без аргументів видаляє рядок за будь-якого пропуску, а в наборі є майже порожні стовпці (ФЕРИТ2 - 1 значення зі 121)',
     'Після dropna() не залишилося жодного рядка (0 зі 121), тож «повністю заповнений набір, придатний для моделі» - це порожня таблиця; звіт сам друкує 0, але висновок стверджує протилежне.',
     'Спершу відкинути стовпці з надмірною часткою пропусків (понад 20%), потім рядки: лишається 65 рядків із 56 стовпцями; або обґрунтовано імпутувати.'),
    (10,
     'Витік цілі й порушені шкали: NaT дати розтину переведено в int64, а Пірсона застосовано до всього підряд',
     'Дата розтину є лише в померлих (61 з 61), а пропуск став числом -9.2e18, тому r = 1.00 відображає саму ціль, а не передбачає її; решта r - це лінійний зв’язок із номінальною міткою, що має сенс лише для бінарних і кількісних ознак.',
     'Виключити ознаки, відомі лише після події (дата розтину, виписка), не переводити NaT у числа, для кількісних ознак брати точково-бісеріальний коефіцієнт або AUC, а для категорій - V Крамера.'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Середнє арифметичне для номінальної (бінарної) змінної: ...,"Значення 1.50 каже тільки те, що померлих і вижилих майж...",Подавати частоти й частки: 61 померлий (50.4%) і 60 вижи...
1,2,Дані непридатні: одну категорію записано по-різному («ж»...,"Чотири «статі» - це дві статі в різних записах, тому вис...","Спершу звести написання до двох значень (ч - 46, ж - 75)..."
2,3,"Середнє для порядкової шкали, до того ж порядок задано а...","Алфавіт ставить «важкий» перед «легкий», а «важкий » із ...","Очистити написання, задати порядок легкий < середній < в..."
3,4,"Дані непридатні: КТ має змішаний формат (частки 0.3, ряд...",Середнє 39.6% пораховано лише за 27 числовими записами з...,Привести всі записи до єдиної частки (діапазон - середин...
4,5,Дані непридатні: у spO2 без дотації є значення 0.9 - ймо...,Живий пацієнт не може мати сатурацію 0.9%; цей викид тяг...,"Перевірити первинний запис, виправити або виключити вики..."
5,6,Коефіцієнт варіації для інтервальної шкали (дати): нуль ...,"Дати переведено в дні від 1970-01-01, тому середнє близь...","Для дат подавати діапазон, медіану та стандартне відхиле..."
6,7,Кореляція Пірсона між двома номінальними змінними (стать...,"r = 0.177 залежить від того, які числа дали категоріям, ...","Таблиця спряженості, χ² та коефіцієнт Крамера V (для 2×2..."
7,8,Мін-макс нормалізація застосована до всіх числових стовп...,"ФЕРИТ2 має єдине значення, max - min = 0, тож ділення на...",Спершу відібрати кількісні ознаки за шкалою й варіативні...
8,9,Дані непридатні: dropna() без аргументів видаляє рядок з...,"Після dropna() не залишилося жодного рядка (0 зі 121), т...",Спершу відкинути стовпці з надмірною часткою пропусків (...
9,10,Витік цілі й порушені шкали: NaT дати розтину переведено...,"Дата розтину є лише в померлих (61 з 61), а пропуск став...","Виключити ознаки, відомі лише після події (дата розтину,..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [23]:
print('РОЗДІЛ 1. Результат лікування')
print('  хибно   : середнє кодів =', round(df['помер/жив'].mean(), 2))
print('  правильно:')
print(df['помер/жив'].map({1: 'помер', 2: 'вижив'}).value_counts().to_string())
print('  летальність:', round(100 * (df['помер/жив'] == 1).mean(), 1), '%')
print()

print('РОЗДІЛ 2. Стать')
print('  хибно   : унікальних значень =', df['Стать'].nunique())
стать_чиста = df['Стать'].str.strip().str[0].map({'ч': 'чоловіча', 'ж': 'жіноча'})
print('  правильно: унікальних значень =', стать_чиста.nunique())
print(стать_чиста.value_counts().to_string())
print(стать_чиста.value_counts(normalize=True).mul(100).round(1).to_string())
print()

print('РОЗДІЛ 3. Ступінь ураження за КТ')
ступінь = df['ступінь КТ'].value_counts().sort_index()
коди = {назва: i for i, назва in enumerate(ступінь.index)}
середній = df['ступінь КТ'].map(коди).mean()
print('  хибно   : середній код', round(середній, 2), '->', ступінь.index[int(round(середній))])
очищення = {'легка': 'легкий', 'легкий': 'легкий', 'сердній': 'середній', 'серед ступ': 'середній',
            'серед ступеня': 'середній', 'середній': 'середній', 'важкий': 'важкий',
            'вкрай важкий': 'вкрай важкий', 'критичний': 'критичний'}
порядок_кт = ['легкий', 'середній', 'важкий', 'вкрай важкий', 'критичний']
ступінь_чистий = pd.Categorical(df['ступінь КТ'].str.strip().map(очищення), categories=порядок_кт, ordered=True)
коди_кт = ступінь_чистий.codes[ступінь_чистий.codes >= 0]
print('  правильно: частоти за порядком тяжкості')
print(pd.Series(ступінь_чистий).value_counts(sort=False).to_string())
print('  медіана:', порядок_кт[int(np.quantile(коди_кт, 0.5, method='lower'))])
print()

print('РОЗДІЛ 7. Стать і результат')
стать_код = стать_чиста.map({'чоловіча': 0, 'жіноча': 1})
print('  хибно   : Пірсон r =', round(стать_код.corr((df['помер/жив'] == 1).astype(int)), 3))
т = pd.crosstab(стать_чиста, df['помер/жив'])
х, p_, _, _ = chi2_contingency(т, correction=False)
print('  правильно: χ² =', round(х, 2), ', p =', round(p_, 3), ', V =', round(np.sqrt(х / т.values.sum()), 3))
print(т.to_string())
print()

print('РОЗДІЛ 9. Очищення від пропусків')
print('  хибно   : рядків після df.dropna() =', len(df.dropna()))
частка_пропусків = df.isna().mean()
придатні = частка_пропусків[частка_пропусків <= 0.2].index
print('  правильно: стовпців із пропусками не більше 20% =', len(придатні),
      ', повних рядків =', len(df[придатні].dropna()), 'з', len(df))

РОЗДІЛ 1. Результат лікування
  хибно   : середнє кодів = 1.5
  правильно:
помер/жив
помер    61
вижив    60
  летальність: 50.4 %

РОЗДІЛ 2. Стать
  хибно   : унікальних значень = 4
  правильно: унікальних значень = 2
Стать
жіноча      75
чоловіча    46
Стать
жіноча      62.0
чоловіча    38.0

РОЗДІЛ 3. Ступінь ураження за КТ
  хибно   : середній код 5.9 -> легкий
  правильно: частоти за порядком тяжкості
легкий           7
середній        18
важкий          10
вкрай важкий     4
критичний        3
  медіана: середній

РОЗДІЛ 7. Стать і результат
  хибно   : Пірсон r = 0.177
  правильно: χ² = 3.78 , p = 0.052 , V = 0.177
помер/жив   1   2
Стать            
жіноча     43  32
чоловіча   18  28

РОЗДІЛ 9. Очищення від пропусків
  хибно   : рядків після df.dropna() = 0
  правильно: стовпців із пропусками не більше 20% = 56 , повних рядків = 65 з 121


---
# Етап 7. Висновки та індивідуальне питання

In [24]:
ВИСНОВКИ = [
    'Стовпець Телефон є ідентифікатором: усі 120 непорожніх значень унікальні, а 23 з них збережено як числа без початкового нуля, тому жодна статистика для нього не має змісту й у модель він не йде.',
    'ЛН є порядковою шкалою, тож коректно описувати її медіаною («ІІ-ІІІ») і квантилями, а середнє кодів безглузде, бо відстані між ступенями невідомі й залежать від довільної нумерації.',
    f'Зв’язок ЛН із результатом лікування сильний: V Крамера = {V:.2f}, χ² = {хі2:.1f}, усі 61 померлі мали ЛН не нижче ІІ-ІІІ, але 4 із 12 клітинок мають очікувану частоту менше 5, тож p наближене.',
    'Дата розтину заповнена лише у померлих (61 з 61, у вижилих жодного значення), тому використання її для прогнозу летальності дає витік цілі, а не прогностичну ознаку.',
    'Стовпці КТ і ФЕРИТ1 непридатні для безпосереднього аналізу: КТ записано в змішаному форматі з діапазонами та 64% пропусків, а ФЕРИТ1 заповнено лише у двох пацієнтів зі 121.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Стовпець Телефон є ідентифікатором: усі 120 непорожніх значень унікальні, а 23 з них збережено як числа без початкового нуля, тому жодна статистика для нього не має змісту й у модель він не йде.
2. ЛН є порядковою шкалою, тож коректно описувати її медіаною («ІІ-ІІІ») і квантилями, а середнє кодів безглузде, бо відстані між ступенями невідомі й залежать від довільної нумерації.
3. Зв’язок ЛН із результатом лікування сильний: V Крамера = 0.76, χ² = 69.6, усі 61 померлі мали ЛН не нижче ІІ-ІІІ, але 4 із 12 клітинок мають очікувану частоту менше 5, тож p наближене.
4. Дата розтину заповнена лише у померлих (61 з 61, у вижилих жодного значення), тому використання її для прогнозу летальності дає витік цілі, а не прогностичну ознаку.
5. Стовпці КТ і ФЕРИТ1 непридатні для безпосереднього аналізу: КТ записано в змішаному форматі з діапазонами та 64% пропусків, а ФЕРИТ1 заповнено лише у двох пацієнтів зі 121.


In [25]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()

тиск = df['АТ пост'].str.extract(r'^\s*(\d+)\s*/\s*(\d+)\s*$').astype(float)
тиск.columns = ['САТ', 'ДАТ']
тиск['пульсовий'] = тиск['САТ'] - тиск['ДАТ']
тиск['середній'] = тиск['ДАТ'] + тиск['пульсовий'] / 3
print(тиск.describe().round(1).T.to_string())
print()

ВІДПОВІДЬ = (
    f'Значення «АТ пост» - це текст «систолічний/діастолічний», тому в модель у такому вигляді він не потрапить; '
    f'регулярний вираз розібрав {тиск["САТ"].notna().sum()} записів зі {len(df)}, а відсутній запис лишається пропуском, а не нулем. '
    'Є три способи. Перший - розділити рядок на два числові стовпці: систолічний (САТ) і діастолічний (ДАТ) тиск у мм рт. ст. '
    'Другий - обчислити похідні показники: пульсовий тиск (САТ - ДАТ) і середній артеріальний тиск (ДАТ + пульсовий / 3). '
    'Третій - віднести пацієнта до категорії за клінічними порогами (норма, гіпертензія 1, 2, 3 ступеня) і закодувати її як порядкову ознаку. '
    'У модель я візьму перший спосіб: два стовпці зберігають усю інформацію, мають шкалу відношень, прості для інтерпретації та перевірки, '
    'і їх можна логарифмувати чи стандартизувати. Третій спосіб губить інформацію (140/100 і 159/109 потраплять в одну категорію), а пороги умовні. '
    'Другий спосіб дає лінійні комбінації тих самих двох чисел, тому САТ, ДАТ і середній тиск разом брати не слід через колінеарність; '
    'пульсовий тиск можна додати окремо лише за наявності клінічного обґрунтування.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «АТ пост» містить значення на кшталт «140/100». Запропонуйте три способи перетворити його на числові ознаки та обґрунтуйте, який із них візьмете в модель.

           count   mean   std   min    25%    50%    75%    max
САТ        120.0  139.9  16.9  90.0  130.0  140.0  150.0  200.0
ДАТ        120.0   88.1  11.6  60.0   80.0   90.0  100.0  120.0
пульсовий  120.0   51.8   8.5  30.0   50.0   50.0   52.5   80.0
середній   120.0  105.4  13.0  70.0   96.7  103.3  116.7  146.7

Значення «АТ пост» - це текст «систолічний/діастолічний», тому в модель у такому вигляді він не потрапить; регулярний вираз розібрав 120 записів зі 121, а відсутній запис лишається пропуском, а не нулем. Є три способи. Перший - розділити рядок на два числові стовпці: систолічний (САТ) і діастолічний (ДАТ) тиск у мм рт. ст. Другий - обчислити похідні показники: пульсовий тиск (САТ - ДАТ) і середній артеріальний тиск (ДАТ + пульсовий / 3). Третій - віднести пацієнта до категорії за клінічними поро

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [26]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, як є',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'текст, як є',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'текст, як є',
    'етап 7 — висновки й відповідь':   'текст, як є',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Я самостійно переглянув увесь код у цьому зошиті та розібрався, як він працює. Добре розумію, як ми замінювали текст на числа, як шукали безглузді значення та рахували кореляцію. На захисті зможу пояснити будь-який рядок і змінити його, якщо скажете.'
ПІДТВЕРДЖУЮ = True

In [27]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 2 готова до захисту.
Голейчук Іван, Ші-33


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.